In [0]:
import os
import time

from databricks import sql

DEFAULT_AS_OF = spark.sql(
    "SELECT date_format(from_utc_timestamp(current_timestamp(), 'America/Toronto'), 'yyyy-MM-dd') AS d"
).first()["d"]

dbutils.widgets.text("catalog", "serverless_nv91sa_catalog")
dbutils.widgets.text("team", "team1")
dbutils.widgets.text("rt_warehouse_id", "")
dbutils.widgets.text("host", spark.conf.get("spark.databricks.workspaceUrl", ""))
dbutils.widgets.text("token", "")
dbutils.widgets.text("as_of_date", DEFAULT_AS_OF)

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")
warehouse_id = dbutils.widgets.get("rt_warehouse_id")
host = dbutils.widgets.get("host").replace("https://", "")
token = dbutils.widgets.get("token")

show_create = spark.sql(f"SHOW CREATE TABLE {catalog}.{team}.gold_order_margin").collect()
clustered = any("CLUSTER BY" in row[0].upper() for row in show_create)

elapsed_ms = -1.0
if warehouse_id and host and token:
    start = time.time()
    with sql.connect(server_hostname=host, http_path=f"/sql/1.0/warehouses/{warehouse_id}", access_token=token) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"SELECT COUNT(*) AS c FROM {catalog}.{team}.gold_order_margin WHERE order_ts >= current_timestamp() - INTERVAL 1 DAY")
            cursor.fetchall()
    elapsed_ms = (time.time() - start) * 1000

status = "PASS" if clustered and elapsed_ms >= 0 else "FAIL"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.ops.milestones (team STRING, round INT, status STRING, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP)")
spark.sql(
    f"INSERT INTO {catalog}.ops.milestones VALUES ('{team}', 2, '{status}', 'round_trip_ms', {elapsed_ms}, current_timestamp())"
)

print({"status": status, "clustered": clustered, "elapsed_ms": elapsed_ms})
